# 66 · Organize direct Oλ GSEA and regenerate publication figures
Use the completed direct-scale, dataset-adjusted pseudobulk-particle enrichment. Render significant Hallmarks, the Reactome overlap network, and the matching community NES summary. Reuse pathway identities, overlap edges, cluster assignments, and the **final refined node coordinates** from the earlier pseudobulk-particle publication figures. All NES values, FDR decisions, node emphasis, label eligibility, and community statistics come from the direct run. Positive NES indicates enrichment toward higher direct Oλ.

Run all cells after enrichment finishes. Outputs are written inside the selected direct run. No gene correlations, enrichment, network clustering, layout refinement, or Reactome hierarchy downloads are needed.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import tempfile
import numpy as np
import pandas as pd
from IPython.display import display

drive.mount('/content/drive')
DATA=Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
BASE=DATA/'bayesian_direct_Olambda_gene_associations_dataset_adjusted_pseudobulk_particles'
OLD=DATA/'bayesian_Olambda_gene_associations_dataset_adjusted_pseudobulk_particles'
NETWORK=OLD/'publication_GSEA_figures'/'Reactome_network'
DIRECT_RUN=None  # Optional exact run directory, if multiple completed full runs are available.

In [ ]:
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def complete(path):
    required=['run_config.json','gene_association_manifest.json','analysis_cohort.csv',
              'olambda_gene_associations.csv','olambda_gene_GSEA_rank.rnk',
              'fgsea/fgsea_collection_versions.csv','fgsea/enrichment_summary.csv']
    return all((path/f).is_file() for f in required) and all(
        (path/'fgsea'/f'{stem}_{name}.csv').is_file()
        for stem in ['membership','fgsea'] for name in ['Hallmark','Reactome'])

paths=[Path(DIRECT_RUN)] if DIRECT_RUN else sorted(BASE.glob('*/'))
candidates=[]
for path in paths:
    if complete(path):
        cfg=json.loads((path/'run_config.json').read_text())
        if cfg['adjustment']=='dataset' and cfg['draws']==4000 and cfg['simple_permutations']==1_000_000:
            candidates.append(path)
if len(candidates)!=1: raise ValueError(f'Expected one completed direct run under {BASE}. Found {candidates}. Set DIRECT_RUN if needed.')
RUN=candidates[0]
FG=RUN/'fgsea'
OUT=RUN/'publication_GSEA_figures'
TABLE=OUT/'tables'; HALL=OUT/'Hallmark'; NET=OUT/'Reactome_network'
for p in [TABLE,HALL,NET]: p.mkdir(exist_ok=True,parents=True)
association=json.loads((RUN/'gene_association_manifest.json').read_text())
assert association['patients']==163 and len(association['datasets'])==8 and association['draws']==4000
assert Path(association['olambda_particles_csv']).parent.name=='olambda_reciprocal_direct_no_asymptote_floor_v1'
print('Direct enrichment:',FG,'\nFigures:',OUT)

## Audit the pathway universe and transfer the saved topology
The full node table supplies cluster assignments for every tested pathway. The published refined node table supplies coordinates for the displayed subset. Use pathway names for both joins.

In [ ]:
versions=pd.read_csv(FG/'fgsea_collection_versions.csv').set_index('collection')
old_versions=pd.read_csv(OLD/'fgsea/fgsea_collection_versions.csv').set_index('collection')
pd.testing.assert_frame_equal(versions.sort_index(),old_versions.sort_index(),check_dtype=False)
for name in ['Hallmark','Reactome']:
    filename=f'membership_{name}.csv'
    def members(path):
        return pd.read_csv(path,usecols=['pathway','gene']).drop_duplicates().sort_values(['pathway','gene']).reset_index(drop=True)
    pd.testing.assert_frame_equal(members(FG/filename),members(OLD/'fgsea'/filename))
    new_set=pd.read_csv(FG/f'fgsea_{name}.csv').set_index('pathway',verify_integrity=True)['size']
    old_set=pd.read_csv(OLD/'fgsea'/f'fgsea_{name}.csv').set_index('pathway',verify_integrity=True)['size']
    pd.testing.assert_series_equal(new_set.sort_index(),old_set.sort_index(),check_dtype=False)

rx=pd.read_csv(FG/'fgsea_Reactome.csv')
nodes_path=NETWORK/'reactome_enrichmentmap_nodes.csv'
layout_path=NETWORK/'olambda_reactome_enrichment_nodes.csv'
edges_path=NETWORK/'reactome_enrichmentmap_edges.csv'
modules_path=NETWORK/'reactome_autoannotate_clusters.csv'
struct=pd.read_csv(nodes_path,usecols=['pathway','node','cluster','displayed','top_topic','official_branch','has_connection'])
assert struct.pathway.is_unique and set(struct.pathway)==set(rx.pathway)
layout=pd.read_csv(layout_path,usecols=['pathway','cluster','x','y'])
assert set(layout.pathway)==set(struct.loc[struct.displayed,'pathway'])
nodes=struct.merge(rx,on='pathway',validate='one_to_one').merge(
    layout,on=['pathway','cluster'],how='left',validate='one_to_one')
assert np.isfinite(nodes.loc[nodes.displayed,['x','y']]).all().all()
nodes['significant']=nodes.padj.lt(.05) & np.isfinite(nodes.NES)
nodes['plot_padj']=nodes.padj.fillna(1)
nodes.to_csv(NET/'direct_reactome_nodes_with_frozen_layout.csv',index=False)
mods=pd.read_csv(modules_path,usecols=['cluster','display_label'])
assert mods.cluster.is_unique and set(nodes.cluster.dropna())<=set(mods.cluster)
mods.to_csv(NET/'reactome_module_labels.csv',index=False)
edges=pd.read_csv(edges_path)
assert set(edges.source_pathway)|set(edges.target_pathway)<=set(nodes.pathway)
shutil.copy2(edges_path,NET/edges_path.name)
stats=nodes.groupby('cluster').agg(pathways=('pathway','size'),finite_NES=('NES','count'),
    significant_pathways=('significant','sum'),mean_NES=('NES','mean'),median_NES=('NES','median'),
    minimum_NES=('NES','min'),maximum_NES=('NES','max'))
stats.join(mods.set_index('cluster')).to_csv(TABLE/'all_Reactome_community_statistics.csv')
for name in ['fgsea_Hallmark.csv','fgsea_Reactome.csv','fgsea_all_collections.csv','enrichment_summary.csv',
             'fgsea_collection_versions.csv','membership_Hallmark.csv','membership_Reactome.csv']:
    shutil.copy2(FG/name,TABLE/name)
for name in ['olambda_gene_associations.csv','olambda_gene_GSEA_rank.rnk','analysis_cohort.csv']:
    shutil.copy2(RUN/name,TABLE/name)
display(pd.read_csv(FG/'enrichment_summary.csv'))
print(len(nodes),'pathways;',len(layout),'fixed positions;',len(mods),'communities')

In [ ]:
REVISION='8ea2e503ada7f849a0a32fb0f77e9eeb9e6638ac'
repo=Path(tempfile.mkdtemp(prefix='direct-gsea66-'))/'repo'
subprocess.run(['git','clone','--quiet','--branch','direct-olambda','--single-branch',
                'https://github.com/adamnik/CAR-T-spectral-organization.git',repo],check=True)
subprocess.run(['git','-C',repo,'checkout','--quiet','--detach',REVISION],check=True)
scripts=repo/'gene_enrichment'
def run(cmd):
    with subprocess.Popen(list(map(str,cmd)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout: print(line,end='',flush=True)
        if proc.wait(): raise subprocess.CalledProcessError(proc.returncode,cmd)
run(['Rscript','-e', """p <- c("igraph","ggraph","ggrepel","ggplot2","scales")
missing <- p[!vapply(p,requireNamespace,logical(1),quietly=TRUE)]
if(length(missing)) install.packages(missing,repos="https://cloud.r-project.org")
stopifnot(all(vapply(p,requireNamespace,logical(1),quietly=TRUE)))"""])
run(['Rscript',scripts/'plot_olambda_hallmark_gsea.R',FG/'fgsea_Hallmark.csv',HALL])
run(['Rscript',scripts/'plot_reactome_enrichment_network.R',NET/'direct_reactome_nodes_with_frozen_layout.csv',
     NET/edges_path.name,NET/'reactome_module_labels.csv',NET,'context','olambda_reactome_enrichment','fixed'])

In [ ]:
plotted=pd.read_csv(NET/'olambda_reactome_enrichment_nodes.csv').set_index('pathway').sort_index()
original=layout.set_index('pathway').sort_index()
assert plotted.index.equals(original.index)
np.testing.assert_allclose(plotted[['x','y']],original[['x','y']],atol=1e-12,rtol=0)
new=nodes.set_index('pathway').loc[plotted.index]
np.testing.assert_allclose(plotted.NES,new.NES,equal_nan=True)
summary=pd.read_csv(NET/'olambda_reactome_enrichment_module_summary.csv').set_index('cluster')
included=new.groupby('cluster').significant.sum()
assert set(summary.index)==set(included[included>=6].index)
np.testing.assert_allclose(summary.mean_NES,new.groupby('cluster').NES.mean().loc[summary.index])
hall=pd.read_csv(FG/'fgsea_Hallmark.csv')
hall_plotted=pd.read_csv(HALL/'hallmark_significant_pathways.csv')
assert set(hall_plotted.pathway)==set(hall.loc[hall.padj.lt(.05)&np.isfinite(hall.NES),'pathway'])
figures=[HALL/'hallmark_significant_enrichment',NET/'olambda_reactome_enrichment',
         NET/'olambda_reactome_enrichment_module_summary']
from PIL import Image
for stem in figures:
    assert stem.with_suffix('.pdf').stat().st_size>0
    with Image.open(stem.with_suffix('.jpg')) as im:
        im.draft('RGB',(1100,850)); im.thumbnail((1100,850)); display(im.copy())
    print(stem.with_suffix('.jpg'))
manifest=dict(direct_run=str(RUN),revision=REVISION,topology='frozen published refined coordinates',
              source_sha256={str(p):sha(p) for p in [nodes_path,layout_path,edges_path,modules_path,
                               FG/'fgsea_Hallmark.csv',FG/'fgsea_Reactome.csv',RUN/'run_config.json']},
              significant_hallmarks=len(hall_plotted),displayed_nodes=len(plotted),labeled_modules=len(summary))
(OUT/'GSEA_figure_manifest.json').write_text(json.dumps(manifest,indent=2))
display(summary[['label','pathways','mean_NES','q10_NES','q90_NES']])